# Experimento Completo: 11 Datasets + 6 Modelos

**Modelos:** GBML, ACDWM, HAT, ARF, SRP, ERulesD2S

**Datasets:** 11 (RBF, SEA, AGRAWAL, HYPERPLANE, STAGGER, SINE)

**Tempo estimado:** 14 horas

**Margem Colab Pro:** 10 horas (42%)

## 1. Setup Inicial

### 1.1 Verificar GPU

In [ ]:
!nvidia-smi
print("\nGPU detectada com sucesso!")

### 1.2 Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive montado!")

### 1.3 Upload dos Arquivos do Projeto

**Opção A:** Fazer upload do ZIP manualmente

**Opção B:** Clonar do GitHub (se disponível)

In [ ]:
# Opção A: Upload manual do ZIP
# 1. Compactar a pasta DSL-AG-hybrid localmente
# 2. Upload para /content/drive/MyDrive/
# 3. Descompactar:

!unzip -q /content/drive/MyDrive/DSL-AG-hybrid.zip -d /content/
%cd /content/DSL-AG-hybrid
!ls -la

# Ou Opção B: Se já tiver no Drive
# !cp -r /content/drive/MyDrive/DSL-AG-hybrid /content/
# %cd /content/DSL-AG-hybrid

### 1.4 Instalar Dependências Python

In [ ]:
!pip install -q river scikit-learn pandas numpy matplotlib seaborn pyyaml xgboost
print("Dependências Python instaladas!")

# Verificar versões
import river
import sklearn
import xgboost
print(f"River: {river.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"XGBoost: {xgboost.__version__}")

## 2. Setup ERulesD2S

### 2.1 Instalar Java e Maven

In [ ]:
%%time
print("Instalando Java 11 e Maven...")
!apt-get update -qq
!apt-get install -y -qq openjdk-11-jdk-headless maven

# Verificar instalação
!java -version
!mvn --version
print("\nJava e Maven instalados com sucesso!")

### 2.2 Executar Setup do ERulesD2S

In [ ]:
%%time
# Setup automático do ERulesD2S
!python setup_erulesd2s.py

# Verificar se JAR foi criado
import os
if os.path.exists('erulesd2s.jar'):
    print("\n[OK] ERulesD2S configurado com sucesso!")
    jar_size = os.path.getsize('erulesd2s.jar') / (1024 * 1024)
    print(f"JAR size: {jar_size:.2f} MB")
else:
    print("\n[ERRO] ERulesD2S JAR não encontrado!")

### 2.3 Teste Rápido do ERulesD2S

In [ ]:
%%time
# Teste unitário rápido
!python test_erulesd2s_integration.py

print("\n[INFO] Teste do ERulesD2S concluído!")

## 3. Configuração do Experimento

### 3.1 Verificar Configuração

In [ ]:
import yaml

# Carregar configuração
with open('config_experiment_expanded.yaml') as f:
    config = yaml.safe_load(f)

datasets = config['experiment_settings']['drift_simulation_experiments']
chunk_size = config['data_params']['chunk_size']
num_chunks = config['data_params']['num_chunks']

print("=" * 80)
print("CONFIGURAÇÃO DO EXPERIMENTO")
print("=" * 80)
print(f"\nDatasets: {len(datasets)}")
for ds in datasets:
    print(f"  - {ds}")
print(f"\nChunk size: {chunk_size}")
print(f"Chunks por dataset: {num_chunks}")
print(f"Total de chunks: {len(datasets) * num_chunks}")
print(f"\nModelos: GBML, ACDWM, HAT, ARF, SRP, ERulesD2S (6)")
print(f"\nTempo estimado: 14 horas")

### 3.2 Script de Keep-Alive

In [ ]:
from IPython.display import display, Javascript

def keep_alive():
    display(Javascript('''
        function KeepClicking(){
            console.log("Keeping session alive...");
            document.querySelector("colab-toolbar-button#connect").click();
        }
        setInterval(KeepClicking, 60000);
    '''))

keep_alive()
print("Keep-alive ativado! Sessão será mantida ativa.")

## 4. Execução do Experimento Completo

### 4.1 Script de Backup Automático

In [ ]:
import threading
import time
import shutil
from pathlib import Path

def backup_results():
    """Faz backup dos resultados a cada hora"""
    while True:
        time.sleep(3600)  # 1 hora
        try:
            results_dir = Path('experiment_expanded_results')
            if results_dir.exists():
                backup_dir = Path('/content/drive/MyDrive/experiment_backup')
                backup_dir.mkdir(parents=True, exist_ok=True)
                
                timestamp = time.strftime('%Y%m%d_%H%M%S')
                backup_file = backup_dir / f'backup_{timestamp}.tar.gz'
                
                !tar -czf {backup_file} experiment_expanded_results/
                print(f"[BACKUP] Resultados salvos: {backup_file}")
        except Exception as e:
            print(f"[BACKUP ERROR] {e}")

# Iniciar thread de backup
backup_thread = threading.Thread(target=backup_results, daemon=True)
backup_thread.start()
print("Backup automático ativado (a cada 1 hora)")

### 4.2 Executar Experimento (FASE PRINCIPAL - 14 HORAS)

In [ ]:
%%time
# IMPORTANTE: Esta célula executa o experimento completo (~14 horas)
# NÃO INTERROMPER durante a execução!

# Executar script de experimento completo
!python run_experiment_complete.py

print("\n" + "=" * 80)
print("EXPERIMENTO CONCLUÍDO!")
print("=" * 80)
print("\nVerifique os resultados nas células abaixo.")

### 4.3 Monitorar Progresso (Executar em paralelo)

In [ ]:
# Executar esta célula periodicamente para ver progresso
import glob
import pandas as pd

results_dir = 'experiment_expanded_results'
csv_files = glob.glob(f'{results_dir}/**/comparison_table.csv', recursive=True)

print("=" * 80)
print("PROGRESSO DO EXPERIMENTO")
print("=" * 80)
print(f"\nDatasets completados: {len(csv_files)}/11")
print(f"Progresso: {len(csv_files)/11*100:.1f}%")

if csv_files:
    print("\nDatasets concluídos:")
    for f in sorted(csv_files):
        dataset_name = f.split('/')[1].split('_seed')[0]
        print(f"  ✓ {dataset_name}")

# Mostrar últimos logs
print("\n" + "=" * 80)
print("ÚLTIMAS LINHAS DO LOG")
print("=" * 80)
!tail -n 20 experiment_expanded_results/latest.log 2>/dev/null || echo "Log não encontrado"

## 5. Consolidação dos Resultados

### 5.1 Consolidar Resultados Python

In [ ]:
import pandas as pd
import glob

# Carregar todos os CSVs
csv_files = glob.glob('experiment_expanded_results/**/comparison_table.csv', recursive=True)

print(f"Encontrados {len(csv_files)} arquivos CSV")

# Consolidar
dfs = []
for csv_file in csv_files:
    df = pd.read_csv(csv_file)
    dataset_name = csv_file.split('/')[1].split('_seed')[0]
    df['dataset'] = dataset_name
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

# Salvar consolidado
df_all.to_csv('experiment_expanded_results_consolidated.csv', index=False)
print(f"\nResultados consolidados: {len(df_all)} linhas")

# Estatísticas
print("\n" + "=" * 80)
print("ESTATÍSTICAS POR MODELO")
print("=" * 80)
summary = df_all.groupby('model')['gmean'].agg(['count', 'mean', 'std', 'min', 'max'])
print(summary.round(4))

print("\n" + "=" * 80)
print("RANKING POR G-MEAN MÉDIO")
print("=" * 80)
ranking = summary['mean'].sort_values(ascending=False)
for idx, (model, gmean) in enumerate(ranking.items(), 1):
    print(f"  {idx}. {model:<12} : {gmean:.4f}")

### 5.2 Adicionar Resultados do ERulesD2S

In [ ]:
# Carregar resultados ERulesD2S (se disponíveis)
erulesd2s_results_file = 'experiment_expanded_results_erulesd2s.csv'

if os.path.exists(erulesd2s_results_file):
    df_erulesd2s = pd.read_csv(erulesd2s_results_file)
    
    # Merge com resultados Python
    df_complete = pd.concat([df_all, df_erulesd2s], ignore_index=True)
    
    # Salvar completo
    df_complete.to_csv('experiment_expanded_results_complete_6_models.csv', index=False)
    
    print("[OK] Resultados ERulesD2S adicionados!")
    print(f"Total de linhas: {len(df_complete)}")
    
    # Ranking atualizado
    print("\n" + "=" * 80)
    print("RANKING FINAL (6 MODELOS)")
    print("=" * 80)
    summary_complete = df_complete.groupby('model')['gmean'].agg(['count', 'mean', 'std'])
    ranking_complete = summary_complete['mean'].sort_values(ascending=False)
    for idx, (model, gmean) in enumerate(ranking_complete.items(), 1):
        print(f"  {idx}. {model:<12} : {gmean:.4f}")
else:
    print("[INFO] Resultados ERulesD2S não encontrados.")
    print("Execute a fase ERulesD2S separadamente se necessário.")

## 6. Backup Final

### 6.1 Copiar Resultados para Drive

In [ ]:
%%time
# Copiar resultados completos
!cp -r experiment_expanded_results /content/drive/MyDrive/
print("[OK] Resultados copiados para Drive")

# Criar arquivo compactado
!tar -czf experiment_expanded_results.tar.gz experiment_expanded_results/
!cp experiment_expanded_results.tar.gz /content/drive/MyDrive/
print("[OK] Backup compactado criado")

# Copiar CSVs consolidados
!cp experiment_expanded_results_consolidated.csv /content/drive/MyDrive/
print("[OK] CSV consolidado copiado")

if os.path.exists('experiment_expanded_results_complete_6_models.csv'):
    !cp experiment_expanded_results_complete_6_models.csv /content/drive/MyDrive/
    print("[OK] CSV completo (6 modelos) copiado")

print("\nTODOS OS RESULTADOS SALVOS NO DRIVE!")

### 6.2 Verificar Integridade

In [ ]:
import os

print("=" * 80)
print("VERIFICAÇÃO FINAL")
print("=" * 80)

# Verificar arquivos no Drive
drive_dir = '/content/drive/MyDrive'

files_to_check = [
    'experiment_expanded_results',
    'experiment_expanded_results.tar.gz',
    'experiment_expanded_results_consolidated.csv',
    'experiment_expanded_results_complete_6_models.csv'
]

for f in files_to_check:
    path = os.path.join(drive_dir, f)
    if os.path.exists(path):
        if os.path.isdir(path):
            n_files = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
            print(f"✓ {f} ({n_files} arquivos)")
        else:
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"✓ {f} ({size_mb:.2f} MB)")
    else:
        print(f"✗ {f} (não encontrado)")

print("\n[OK] Verificação concluída!")

## 7. Análise Estatística

### 7.1 Executar Análise Completa

In [ ]:
# Análise estatística completa
# (Adaptar statistical_analysis.py para 11 datasets)

# TODO: Criar script de análise expandida
print("Análise estatística será executada localmente após download dos resultados.")
print("Arquivos necessários estão salvos no Drive.")

## 8. Resumo Final

In [ ]:
from datetime import datetime

print("=" * 80)
print("EXPERIMENTO COMPLETO FINALIZADO")
print("=" * 80)
print(f"\nData/hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nResumo:")
print("  - Datasets: 11")
print("  - Modelos: 6 (GBML, ACDWM, HAT, ARF, SRP, ERulesD2S)")
print("  - Avaliações por modelo: 33")
print("  - Total de avaliações: 198")
print("\nResultados salvos em:")
print("  - /content/drive/MyDrive/experiment_expanded_results/")
print("  - /content/drive/MyDrive/experiment_expanded_results.tar.gz")
print("  - /content/drive/MyDrive/experiment_expanded_results_complete_6_models.csv")
print("\nPróximos passos:")
print("  1. Download dos resultados do Drive")
print("  2. Análise estatística completa")
print("  3. Gerar relatório final")
print("  4. Preparar artigo científico")
print("\n" + "=" * 80)
print("SUCESSO!")
print("=" * 80)